# Lab 2: Train, compare, register, and use a milestone-delay model

Train six classifiers in two rounds in one AIDP experiment, compare their validation results in **Experiments** or in the notebook, register the selected model in the Master Catalog, then reload that exact catalog version to score Gold milestones.

Candidates are ranked by validation F1. When F1 is tied, the lower validation Brier score wins.

Allow 60–90 minutes, including compute startup. Experiments and Models are AIDP Preview features.

### What you will learn

Create an experiment; log parameters, metrics, and model artifacts; compare two model families; select a winner using validation F1 and Brier score; register it in the catalog; and use that registered version for inference.

| Stage | AIDP object or output |
| --- | --- |
| Both training rounds | Experiment `seer_milestone_delay` |
| Round one | Three Decision Tree runs: `dt_r1_1` to `dt_r1_3` |
| Round two | Three Gradient Boosting runs: `gb_r2_1` to `gb_r2_3` |
| Selected model | `default.seer_gold.milestone_delay_classifier` |
| Inference | Gold milestone scores from the selected catalog version |

### How to run this lab

Use workspace **HOL_Workspace** and open the notebook from the Git checkout. Run one Python cell at a time, from top to bottom, and wait for it to finish. Task numbers are Markdown headings, not Python execution counts. This version has **10 Python tasks** and ends after Task 10.

For the first walkthrough, stop after Task 6 to inspect the first model family before running Task 7.

## Prerequisites

1. Complete Lab 1A, Lab 1B, and Lab 1C in order.
2. Ensure Gold contains `milestone_training` and `milestone_features`.
3. Open **aiw_hol → aidp-livelabs → notebooks → Lab2_Predict_Investigate_and_Operationalize.ipynb**.
4. Attach **seer_livelabs_spark**. Use AMD/Intel compute, not ARM.
5. Use the platform-provided MLflow integration. Gradient Boosting is supplied by scikit-learn; no XGBoost installation is required.
6. Your identity needs access to workspace experiments and permission to register model versions in `default.seer_gold`.

`milestone_training` supplies the labeled history for model training. `milestone_features` supplies the current Gold rows for inference.

## Rerunning Lab 2

Do not delete experiments before rerunning. Task 1 creates, reuses or restores `seer_milestone_delay` and starts a new `SESSION_ID`, keeping earlier walkthroughs separate. Run Task 1 only at the start of a complete walkthrough, not between training rounds.

Within the same session, repeating Task 5 or Task 7 reuses completed candidates with identical settings. Failed/killed attempts can be retried; a RUNNING attempt or a completed run without its model URI requires inspection. Keep data, the split and candidate definitions unchanged within a session; restart from Task 1 after changing them.

Task 9 reuses an existing model version for the selected run instead of registering it again. Use the notebook route so Task 10 receives the exact version URI. Pending or failed registrations require inspection. Task 10 can display scores again without training or registering.

These checks protect sequential retries, not simultaneous executions. Run one cell at a time. They do not recover manually deleted model artifacts. Start a fresh session when adopting this version: older runs lack the candidate-key tag. Every complete walkthrough intentionally creates new runs and, for its new winning run, a new model version.


## Task 1: Create the experiment

Run the cell to create, reuse or restore a soft-deleted `seer_milestone_delay`. It also defines the fully qualified model name and creates a session ID for this walkthrough.

**Checkpoint:** Open **Experiments → seer_milestone_delay**. It may be empty until Task 5.


In [ ]:
import json, uuid
from pathlib import Path

import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

utilities = globals().get("oidlUtils")
config_path = "/Workspace/aiw_hol/aidp-livelabs/config/workshop.json"
if utilities is not None:
    config_path = utilities.parameters.getParameter("config_path", config_path)
config = json.loads(Path(config_path).read_text())

GOLD = f"{config['catalog']}.{config['schemas']['gold']}"
TRAINING_TABLE = f"{GOLD}.milestone_training"
SCORING_TABLE = f"{GOLD}.milestone_features"
MODEL_NAME = f"{GOLD}.milestone_delay_classifier"
FEATURES = ["days_to_planned_date", "open_po_fraction", "inspection_fail_flag",
            "inspection_info_flag", "supplier_review_flag", "unresolved_supplier_flag",
            "milestone_not_started_flag", "milestone_at_risk_flag", "log_committed_cost"]

client = MlflowClient()
EXPERIMENT_NAME = "seer_milestone_delay"

# Do not nest a new walkthrough inside an unfinished active run.
if mlflow.active_run() is not None:
    raise RuntimeError(
        "An MLflow run is still active. Finish or resolve that run "
        "before starting another walkthrough."
    )

existing = client.get_experiment_by_name(EXPERIMENT_NAME)
if existing is not None and existing.lifecycle_stage == "deleted":
    client.restore_experiment(existing.experiment_id)
    print("Restored experiment:", EXPERIMENT_NAME)

# Creates a missing experiment or selects the existing active experiment.
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.autolog(disable=True)
# A new session is intentional when restarting the complete walkthrough.
SESSION_ID = uuid.uuid4().hex
threshold = 0.5

print("Experiment:", experiment.name)
print("Session:", SESSION_ID)


## Task 2: Load Gold training history

Run the cell to load the labeled Gold training data and the nine model inputs.


In [ ]:
history = spark.table(TRAINING_TABLE).select(
    "milestone_id", "prediction_as_of", "late_flag", *FEATURES).toPandas()
history["prediction_as_of"] = pd.to_datetime(history["prediction_as_of"], utc=True)
history["late_flag"] = history["late_flag"].astype(int)
history[FEATURES] = history[FEATURES].astype(float)

print({"training_rows": len(history), "late_flag_counts": history.late_flag.value_counts().to_dict()})


## Task 3: Create the train/validation split

Run the cell to create a simple balanced split for the model comparison. Every candidate uses the same split.


In [ ]:
train, validation = train_test_split(
    history, test_size=0.5, random_state=42, stratify=history["late_flag"])

X_train, y_train = train[FEATURES], train.late_flag
X_validation, y_validation = validation[FEATURES], validation.late_flag
print({"training_rows": len(train), "validation_rows": len(validation)})


## Task 4: Define the experiment helper

Run the cell to define the helpers. `train_and_log` reuses a completed matching candidate in this session, or creates one MLflow run and logs its validation metrics. Its candidate key includes the algorithm, parameters, feature list, threshold, round and training-code version. `compare_runs` ranks this walkthrough's runs by validation F1, then validation Brier score.


In [ ]:
import hashlib

# Increment this if the training or evaluation implementation changes.
TRAINING_CODE_VERSION = "1"

def validation_metrics(labels, probabilities):
    predicted = probabilities >= threshold
    return {
        "average_precision": average_precision_score(labels, probabilities),
        "roc_auc": roc_auc_score(labels, probabilities),
        "precision": precision_score(labels, predicted, zero_division=0),
        "recall": recall_score(labels, predicted, zero_division=0),
        "f1": f1_score(labels, predicted, zero_division=0),
        "brier_score": brier_score_loss(labels, probabilities),
    }

def train_and_log(run_name, algorithm, round_number, parameters):
    # Settings identify a candidate within this session. Keep data and split fixed.
    candidate_settings = {
        "run_name": run_name,
        "algorithm": algorithm,
        "round": round_number,
        "parameters": parameters,
        "random_state": 42,
        "threshold": threshold,
        "features": FEATURES,
        "code_version": TRAINING_CODE_VERSION,
    }
    candidate_key = hashlib.sha256(
        json.dumps(candidate_settings, sort_keys=True).encode()
    ).hexdigest()
    candidate_filter = (
        f"tags.session_id = '{SESSION_ID}' AND "
        f"tags.candidate_key = '{candidate_key}'"
    )

    completed = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=candidate_filter + " AND attributes.status = 'FINISHED'",
        max_results=1,
    )
    if completed:
        previous = completed[0]
        if not previous.data.tags.get("model_uri"):
            raise RuntimeError(
                f"Run {previous.info.run_id} finished without a model_uri. "
                "Inspect it before continuing."
            )
        print(f"Reusing {run_name}: {previous.info.run_id}")
        return previous.info.run_id

    # Do not silently duplicate an execution that may still be running.
    running = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=candidate_filter + " AND attributes.status = 'RUNNING'",
        max_results=1,
    )
    if running:
        raise RuntimeError(
            f"Candidate {run_name} already has a RUNNING execution: "
            f"{running[0].info.run_id}. Check it before retrying."
        )
    if mlflow.active_run() is not None:
        raise RuntimeError("Another MLflow run is active in this notebook.")

    # Failed or killed attempts do not prevent a new attempt.
    estimator_class = GradientBoostingClassifier if algorithm == "Gradient Boosting" else DecisionTreeClassifier
    estimator = estimator_class(random_state=42, **parameters)

    with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name) as run:
        mlflow.set_tags({
            "session_id": SESSION_ID,
            "candidate_key": candidate_key,
            "round": str(round_number),
            "algorithm": algorithm,
            "training_code_version": TRAINING_CODE_VERSION,
        })
        mlflow.log_params(parameters)
        estimator.fit(X_train, y_train)
        probabilities = estimator.predict_proba(X_validation)[:, list(estimator.classes_).index(1)]
        metrics = validation_metrics(y_validation, probabilities)
        mlflow.log_metrics({f"validation_{name}": value for name, value in metrics.items()})
        model_info = mlflow.sklearn.log_model(
            estimator, name="model", input_example=X_train.head(3),
            signature=infer_signature(X_train, estimator.predict_proba(X_train)),
            pyfunc_predict_fn="predict_proba")
        mlflow.set_tag("model_uri", model_info.model_uri)
        formatted_metrics = ", ".join(
            f"validation_{name} = {value:.4f}" for name, value in metrics.items()
        )
        print(f"{run_name}: {formatted_metrics}")
    return run.info.run_id

def compare_runs():
    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string=f"tags.session_id = '{SESSION_ID}'")
    return runs.loc[runs.status == "FINISHED"].sort_values(
        ["metrics.validation_f1", "metrics.validation_brier_score", "tags.mlflow.runName"],
        ascending=[False, True, True]).reset_index(drop=True)

comparison_columns = [
    "run_id", "tags.mlflow.runName", "tags.algorithm",
    "metrics.validation_f1", "metrics.validation_brier_score",
    "metrics.validation_average_precision", "metrics.validation_roc_auc",
    "metrics.validation_precision", "metrics.validation_recall",
]

comparison_labels = {
    "run_id": "Run ID",
    "tags.mlflow.runName": "Run name",
    "tags.algorithm": "Algorithm",
    "metrics.validation_average_precision": "Validation average precision",
    "metrics.validation_roc_auc": "Validation ROC AUC",
    "metrics.validation_precision": "Validation precision",
    "metrics.validation_recall": "Validation recall",
    "metrics.validation_f1": "Validation F1",
    "metrics.validation_brier_score": "Validation Brier score (lower is better)",
}

def format_comparison(runs):
    return runs[comparison_columns].rename(columns=comparison_labels).round(4)


## Task 5: Round one — Decision Tree

Run the three Decision Tree candidates.

| Candidate | max_depth | min_samples_leaf |
| --- | --- | --- |
| dt_r1_1 | 3 | 10 |
| dt_r1_2 | 5 | 20 |
| dt_r1_3 | 8 | 30 |

Each configuration creates a separate run in the same experiment. Repeating this cell with unchanged settings in the same session reuses its completed runs.


In [ ]:
tree_candidates = [
    {"max_depth": 3, "min_samples_leaf": 10},
    {"max_depth": 5, "min_samples_leaf": 20},
    {"max_depth": 8, "min_samples_leaf": 30},
]
round_one_ids = [train_and_log(f"dt_r1_{index}", "Decision Tree", 1, parameters)
                 for index, parameters in enumerate(tree_candidates, start=1)]


## Task 6: Compare the first three runs

Run the comparison cell, then open **Experiments → seer_milestone_delay**. Use **List** to inspect parameters and metrics, and **Compare** to visualize `validation_f1` and `validation_brier_score`. The notebook table includes only the runs created in this walkthrough.

Do not choose a winner yet—Task 7 adds the Gradient Boosting runs.


In [ ]:
initial_comparison = compare_runs()
display(format_comparison(initial_comparison))


## Task 7: Round two — Gradient Boosting

Run the three Gradient Boosting candidates.

| Candidate | max_depth | learning_rate | n_estimators |
| --- | --- | --- | --- |
| gb_r2_1 | 2 | 0.05 | 60 |
| gb_r2_2 | 3 | 0.10 | 100 |
| gb_r2_3 | 4 | 0.05 | 120 |

Keep the same experiment and session. Repeating this cell with unchanged settings reuses its completed runs. These runs can now be compared with the Decision Tree runs.


In [ ]:
gb_candidates = [
    {"max_depth": 2, "learning_rate": 0.05, "n_estimators": 60},
    {"max_depth": 3, "learning_rate": 0.10, "n_estimators": 100},
    {"max_depth": 4, "learning_rate": 0.05, "n_estimators": 120},
]
round_two_ids = [train_and_log(f"gb_r2_{index}", "Gradient Boosting", 2, parameters)
                 for index, parameters in enumerate(gb_candidates, start=1)]


## Task 8: Compare all six and select the winner

Run the cell to compare all six candidates. The highest validation F1 wins; when F1 is tied, the lower validation Brier score wins.

You can make the same selection in **Experiments → seer_milestone_delay → Compare**: prefer higher `validation_f1`, then lower `validation_brier_score`.


In [ ]:
final_comparison = compare_runs()
display(format_comparison(final_comparison))

best_run = final_comparison.iloc[0]
best_run_id = best_run["run_id"]
best_model_uri = mlflow.get_run(best_run_id).data.tags["model_uri"]
print("Selected run:", best_run_id)
print("Selection rule: highest validation F1, then lowest validation Brier score")
print(
    f"Selected metrics: F1 = {best_run['metrics.validation_f1']:.4f}, "
    f"Brier score = {best_run['metrics.validation_brier_score']:.4f}"
)


## Task 9: Register the selected model

Run the notebook cell to register the selected F1/Brier winner as `default.seer_gold.milestone_delay_classifier`. It first checks that the winner is a completed run from this session.

If this model already has a version from the selected run, the cell reuses that version. Otherwise, it creates a new one. This lab logs one model per run. Pending or failed versions require inspection rather than another registration attempt.

Use this notebook route for the walkthrough; do not separately register through the UI. The cell defines `REGISTERED_URI` for Task 10. In **Master catalog → default → seer_gold → Models**, open `milestone_delay_classifier` and use the version's **Run** link to trace it to the selected experiment run.


In [ ]:
# Reject a stale winner left over from an earlier walkthrough.
selected_run = client.get_run(best_run_id)
if (
    selected_run.info.experiment_id != experiment.experiment_id
    or selected_run.data.tags.get("session_id") != SESSION_ID
    or selected_run.info.status != "FINISHED"
):
    raise RuntimeError(
        "The selected winner is not a completed run from this session. "
        "Rerun Task 8 before registering."
    )

best_model_uri = selected_run.data.tags["model_uri"]

# This lab logs one model per run. Reuse its existing catalog registration.
existing_versions = client.search_model_versions(
    filter_string=f"name = '{MODEL_NAME}' AND run_id = '{best_run_id}'",
    max_results=1,
)
if existing_versions:
    registered_version = client.get_model_version(
        name=MODEL_NAME, version=existing_versions[0].version
    )
    print("Reusing existing model version:", registered_version.version)
else:
    registered_version = mlflow.register_model(
        model_uri=best_model_uri, name=MODEL_NAME
    )
    print("Created model version:", registered_version.version)

# Pending/failed registrations must be inspected, not duplicated.
registered_version = client.get_model_version(
    name=MODEL_NAME, version=registered_version.version
)
if registered_version.status != "READY":
    raise RuntimeError(
        f"Model version {registered_version.version} has status "
        f"{registered_version.status}. Check it before scoring."
    )

MODEL_VERSION = str(registered_version.version)
REGISTERED_URI = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
print("Registered model:", MODEL_NAME)
print("Version:", MODEL_VERSION, "| Source run:", best_run_id)
print("Versioned model URI:", REGISTERED_URI)


## Task 10: Load the registered version and score Gold milestones

Run the cell to load the exact catalog version created or reused in Task 9 and score `seer_gold.milestone_features`.


In [ ]:
from pyspark.sql import functions as F

loaded_model = mlflow.sklearn.load_model(REGISTERED_URI)
scoring_rows = spark.table(SCORING_TABLE)
positive_class_index = list(loaded_model.classes_).index(1)
predict_proba = mlflow.pyfunc.spark_udf(
    spark, model_uri=REGISTERED_URI, result_type="array<double>", env_manager="local")

predictions = scoring_rows.select(
    "milestone_id", "project_name", "milestone_name", "planned_date",
    predict_proba(F.struct(*[F.col(name).cast("double").alias(name) for name in FEATURES]))[
        positive_class_index
    ].alias("delay_probability"),
).withColumn("predicted_late", F.col("delay_probability") >= threshold) \
 .withColumn("model_version", F.lit(MODEL_VERSION))

print("Scoring with:", REGISTERED_URI)
display(predictions.orderBy(F.desc("delay_probability")))
